# Format Inspection Notebook
**Purpose:** Inspect all raw datasets BEFORE building loaders.

**Critical:** Never assume column names or formats!

In [1]:
import pandas as pd
import os
from pathlib import Path

In [2]:
def inspect_csv_dataset(filepath: str, name: str, max_rows: int = 5):
    """Inspect CSV dataset format."""
    print(f"\n{'='*80}")
    print(f"Dataset: {name}")
    print(f"Path: {filepath}")
    print(f"{'='*80}\n")
    
    if not os.path.exists(filepath):
        print(f"❌ FILE NOT FOUND: {filepath}")
        return None
    
    try:
        df = pd.read_csv(filepath)
        
        print(f"✅ Successfully loaded")
        print(f"\n📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
        print(f"\n📋 Column Names:")
        for i, col in enumerate(df.columns, 1):
            print(f"  {i}. {col} (dtype: {df[col].dtype})")
        
        print(f"\n📝 Sample Data (first {max_rows} rows):")
        print(df.head(max_rows))
        
        # Check for nulls
        null_counts = df.isnull().sum()
        if null_counts.sum() > 0:
            print(f"\n⚠️  Null Values Detected:")
            print(null_counts[null_counts > 0])
        else:
            print(f"\n✅ No null values found")
        
        # Check content lengths for text columns
        text_cols = df.select_dtypes(include=['object']).columns
        if len(text_cols) > 0:
            print(f"\n📏 Content Length Stats (text columns):")
            for col in text_cols:
                if col in df.columns:
                    lengths = df[col].astype(str).str.len()
                    print(f"  {col}: avg={lengths.mean():.1f}, min={lengths.min()}, max={lengths.max()}")
        
        # Sample body content
        body_cols = [col for col in df.columns if 'body' in col.lower() or 'message' in col.lower() or 'content' in col.lower()]
        if body_cols:
            print(f"\n📧 Sample Body Content (first 200 chars):")
            for idx in range(min(2, len(df))):
                content = str(df[body_cols[0]].iloc[idx])[:200]
                print(f"\n  Email {idx+1}: {content}...")
        
        return df
        
    except Exception as e:
        print(f"❌ Error loading file: {e}")
        return None

## 1. Plain LLM Phishing Dataset

In [3]:
plain_llm_df = inspect_csv_dataset(
    'data/raw/plain_llm_phishing.csv',
    'Plain LLM Phishing'
)


Dataset: Plain LLM Phishing
Path: data/raw/plain_llm_phishing.csv

✅ Successfully loaded

📊 Shape: 50 rows × 5 columns

📋 Column Names:
  1. id (dtype: int64)
  2. subject (dtype: object)
  3. sender (dtype: object)
  4. body (dtype: object)
  5. expected_cues (dtype: int64)

📝 Sample Data (first 5 rows):
   id                                 subject  \
0   1     URGENT: Your Account Will Be Closed   
1   2  Confirm Your Email Address Immediately   
2   3            You've Won $5,000 Gift Card!   
3   4        Security Alert: Password Expired   
4   5            Your Package Delivery Failed   

                             sender  \
0  security@paypa1-verification.com   
1       no-reply@amazn-services.net   
2       winners@walmart-rewards.org   
3     alerts@microsoft-security.net   
4         delivery@ups-tracking.org   

                                                body  expected_cues  
0  Dear Valued Customer,\r\n\r\nYour account has ...              5  
1  Hello,\r\n\r\nWe no

## 2. Hybrid V-Triad Phishing Dataset

In [4]:
hybrid_df = inspect_csv_dataset(
    'data/raw/hybrid_vtriad_phishing.csv',
    'Hybrid V-Triad Phishing'
)


Dataset: Hybrid V-Triad Phishing
Path: data/raw/hybrid_vtriad_phishing.csv

✅ Successfully loaded

📊 Shape: 50 rows × 6 columns

📋 Column Names:
  1. id (dtype: int64)
  2. subject (dtype: object)
  3. sender (dtype: object)
  4. body (dtype: object)
  5. expected_cues (dtype: int64)
  6. vtriad_tactic (dtype: object)

📝 Sample Data (first 5 rows):
   id                                    subject                  sender  \
0   1         Your account verification required     security@paypal.com   
1   2                 Confirm recent transaction        alerts@chase.com   
2   3         Complete your profile verification     verify@linkedin.com   
3   4  Action required: Review security settings  security@microsoft.com   
4   5    Confirm shipping address for your order     shipping@amazon.com   

                                                body  expected_cues  \
0  Hello,\r\n\r\nWe've updated our security polic...              2   
1  Dear Customer,\r\n\r\nWe noticed a transaction

## 3. Phishbowl Dataset

In [5]:
phishbowl_df = inspect_csv_dataset(
    'data/raw/phishbowl.csv',
    'Phishbowl Real Phishing'
)


Dataset: Phishbowl Real Phishing
Path: data/raw/phishbowl.csv

✅ Successfully loaded

📊 Shape: 240 rows × 4 columns

📋 Column Names:
  1. Unnamed: 0 (dtype: int64)
  2. title (dtype: object)
  3. date_time (dtype: object)
  4. email_message (dtype: object)

📝 Sample Data (first 5 rows):
   Unnamed: 0                                       title  \
0           0  Regarding Your Request for Account Closure   
1           1                         Students Employment   
2           2                   Baby grand piano donation   
3           3                                         Job   
4           4         STUDENT RESEARCH ASSISTANT POSITION   

                 date_time                                      email_message  
0  Thu, 04/25/2024 - 12:00  <div class="accent-blue-green dialog dialog-no...  
1  Thu, 04/25/2024 - 12:00  <div class="accent-blue-green dialog dialog-no...  
2  Mon, 04/22/2024 - 12:00  <div class="accent-blue-green dialog dialog-no...  
3  Wed, 04/03/2024 - 12:

## 4. SpamAssassin Ham Emails

**Note:** SpamAssassin emails are raw RFC822 files in `data/raw/spamassassin/easy_ham/easy_ham/`

In [6]:
# Inspect SpamAssassin raw files
spam_dir = Path('data/raw/spamassassin/easy_ham/easy_ham')

print(f"\n{'='*80}")
print(f"Dataset: SpamAssassin Ham")
print(f"Path: {spam_dir}")
print(f"{'='*80}\n")

if spam_dir.exists():
    files = list(spam_dir.glob('*'))
    print(f"✅ Directory found")
    print(f"📊 Total files: {len(files)}")
    
    # Sample first file
    if files:
        print(f"\n📧 Sample Email (first file: {files[0].name}):")
        print(f"\n{'-'*80}")
        with open(files[0], 'r', encoding='utf-8', errors='ignore') as f:
            content = f.read()
            print(content[:800])  # First 800 chars
        print(f"\n{'-'*80}")
        
        # Check for blank line separator
        if '\n\n' in content:
            header_section = content.split('\n\n')[0]
            body_section = content.split('\n\n', 1)[1] if len(content.split('\n\n')) > 1 else ""
            print(f"\n✅ Blank line separator found")
            print(f"📏 Header section: {len(header_section)} chars")
            print(f"📏 Body section: {len(body_section)} chars")
            print(f"\n📧 Body preview (first 200 chars):")
            print(body_section[:200])
        else:
            print(f"\n⚠️  No blank line separator found - may need custom parsing")
else:
    print(f"❌ Directory not found: {spam_dir}")


Dataset: SpamAssassin Ham
Path: data\raw\spamassassin\easy_ham\easy_ham

✅ Directory found
📊 Total files: 2551

📧 Sample Email (first file: 0001.ea7e79d3153e7469e7a9c3e0af6a357e):

--------------------------------------------------------------------------------
From exmh-workers-admin@redhat.com  Thu Aug 22 12:36:23 2002
Return-Path: <exmh-workers-admin@example.com>
Delivered-To: zzzz@localhost.netnoteinc.com
Received: from localhost (localhost [127.0.0.1])
	by phobos.labs.netnoteinc.com (Postfix) with ESMTP id D03E543C36
	for <zzzz@localhost>; Thu, 22 Aug 2002 07:36:16 -0400 (EDT)
Received: from phobos [127.0.0.1]
	by localhost with IMAP (fetchmail-5.9.0)
	for zzzz@localhost (single-drop); Thu, 22 Aug 2002 12:36:16 +0100 (IST)
Received: from listman.example.com (listman.example.com [66.187.233.211]) by
    dogma.slashnull.org (8.11.6/8.11.6) with ESMTP id g7MBYrZ04811 for
    <zzzz-exmh@example.com>; Thu, 22 Aug 2002 12:34:53 +0100
Received: from listman.example.com (localhost.locald

## Summary Report

In [7]:
print("\n" + "="*80)
print("DATASET FORMAT SUMMARY")
print("="*80 + "\n")

datasets_info = [
    {
        'name': 'Plain LLM Phishing',
        'df': plain_llm_df,
        'expected_count': 50,
        'class': 'Phishing (1)'
    },
    {
        'name': 'Hybrid V-Triad Phishing',
        'df': hybrid_df,
        'expected_count': 50,
        'class': 'Phishing (1)'
    },
    {
        'name': 'Phishbowl Real Phishing',
        'df': phishbowl_df,
        'expected_count': 50,
        'class': 'Phishing (1)'
    }
]

for info in datasets_info:
    df = info['df']
    if df is not None:
        status = "✅" if len(df) >= info['expected_count'] else "⚠️"
        print(f"{status} {info['name']}:")
        print(f"   Rows: {len(df)} (expected: {info['expected_count']})")
        print(f"   Columns: {', '.join(df.columns.tolist())}")
        print(f"   Class: {info['class']}\n")
    else:
        print(f"❌ {info['name']}: Failed to load\n")

print(f"SpamAssassin Ham: {len(files) if 'files' in locals() else 0} files available (need to extract 100)\n")
print("="*80)


DATASET FORMAT SUMMARY

✅ Plain LLM Phishing:
   Rows: 50 (expected: 50)
   Columns: id, subject, sender, body, expected_cues
   Class: Phishing (1)

✅ Hybrid V-Triad Phishing:
   Rows: 50 (expected: 50)
   Columns: id, subject, sender, body, expected_cues, vtriad_tactic
   Class: Phishing (1)

✅ Phishbowl Real Phishing:
   Rows: 240 (expected: 50)
   Columns: Unnamed: 0, title, date_time, email_message
   Class: Phishing (1)

SpamAssassin Ham: 2551 files available (need to extract 100)



## Export Format Report to File

In [ ]:
# This will be used to create data_inventory.md
print("✅ Format inspection complete. Ready to proceed with Phase 1 data extraction.")